In [ ]:
import sys
sys.path.append("..")

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from src.fdm.method import BTCS
from src.fdm.animation import ANI

In [ ]:
plt.style.use("ggplot")

### **FINITE DIFFERENCE METHOD: BT-CS**

In [ ]:
# dominio espacial: Ω = (0, Lx) x (0, Ly)
Lx = 1
Ly = 1

# intervalo temporal: I = [0, T]
T = 1

# coeficiente de difusión: beta
beta = 1

# término fuente: f(x, y, t)
m = 4
n = 4
def f(x, y, t):
    return beta * np.pi**2 * ((m / Lx)**2 + (n / Ly)**2) * np.cos(m * np.pi * x / Lx) * np.cos(n * np.pi * y / Ly)

# condición inicial: u(x, y, 0) = u0(x, y), (x, y) ∈ Ω 
def u0(x, y):
    return np.float64(0)

# condiciones de contorno: ∂u/∂n = 0, (x, y) ∈ ∂Ω, t ≥ 0

# solución exacta: u(x, y, t)
def u(x, y, t):
    return (1 - np.exp(- beta * np.pi**2 * ((m / Lx)**2 + (n / Ly)**2) * t)) * np.cos(m * np.pi * x / Lx) * np.cos(n * np.pi * y / Ly)

# estado estacionario: u_ss(x, y)
def u_ss(x, y):
    return np.cos(m * np.pi * x / Lx) * np.cos(n * np.pi * y / Ly)

In [ ]:
Nx = 64
Ny = 64
Nt = 128

x, y, t, U = BTCS(f, beta, Lx, Ly, 0.015, u0, Nx, Ny, Nt)
ANI(x, y, U, 0.015, interval=50, title="Ecuación del Calor - BT-CS")

In [ ]:
NN = [16, 32, 64, 128, 256]

h = []
dt = []
err = []

for N in NN:

    Nx = N
    Ny = N
    Nt = N

    x, y, t, U = BTCS(f, beta, Lx, Ly, T, u0, Nx, Ny, Nt)

    X, Y = np.meshgrid(x, y, indexing="ij")

    h.append(max(Lx, Ly) / N)
    dt.append(T / Nt)
    err.append(np.max(np.abs(u(X, Y, T) - U[:, -1].reshape((Nx+1, Ny+1), order="F"))))

p = [None] +[np.log(err[i] / err[i+1]) / np.log(2) for i in range(len(err) - 1)]   

print("-" * 60)
print(f"{"N":>6} | {"h":>10} | {"dt":>10} | {"error":>12} | {"p":>6}")
print("-" * 60)
for i, N in enumerate(NN):
    if (i == 0):
        print(f"{N:6d} | {h[i]:10.5f} | {dt[i]:10.5f} | {err[i]:12.3e} | {'-':>6}")
    else:
        print(f"{N:6d} | {h[i]:10.5f} | {dt[i]:10.5f} | {err[i]:12.3e} | {p[i]:6.2f}")
print("-" * 60)

h = np.array(h)
dt = np.array(dt)
err = np.array(err)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

plt.suptitle("Análisis de Error - BT-CS")

ax[0].loglog(h, err, "o-", color="tomato", label=r"$\|u - U\|_\infty$")
ax[0].loglog(h, err[0] * (h / h[0])**2, "--", color="black", label=r"$\mathcal{O}(h^2)$")
ax[0].set_title(r"$error$ vs $h$")
ax[0].set_xlabel(r"$h$")
ax[0].set_ylabel(r"$error$")
ax[0].invert_xaxis()
ax[0].legend()

ax[1].loglog(dt, err, "o-", color="tomato", label=r"$\|u - U\|_\infty$")
ax[1].loglog(dt, err[0] * (dt / dt[0]), "--", color="black", label=r"$\mathcal{O}(\Delta t)$")
ax[1].set_title(r"$error$ vs $\Delta t$")
ax[1].set_xlabel(r"$\Delta t$")
ax[1].set_ylabel(r"$error$")
ax[1].invert_xaxis()
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
NN = [16, 32, 64, 128, 256]

tm = []

for N in NN:

    Nx = N
    Ny = N
    Nt = N
    
    t0 = time.time()
    BTCS(f, beta, Lx, Ly, T, u0, Nx, Ny, Nt)
    tf = time.time()
    
    tm.append(tf - t0)

plt.figure(figsize=(6, 4))
plt.plot(NN, tm, "o-", color="forestgreen")
plt.title("Costo Computacional - BT-CS")
plt.xlabel(r"$N$")
plt.ylabel(r"$time$ [s]")
plt.xticks(NN)
plt.show()

In [ ]:
Nx = 128
Ny = 128
Nt = 128

x, y, t, U = BTCS(f, beta, Lx, Ly, T, u0, Nx, Ny, Nt)

X, Y = np.meshgrid(x, y, indexing="ij")

U_ss = u_ss(X, Y).reshape(-1, order="F")

err = np.zeros(Nt+1)
for k in range(Nt+1):
    err[k] = np.linalg.norm(U_ss - U[:, k], ord=np.inf)

plt.figure(figsize=(6, 4))
plt.semilogy(t, err, color="steelblue", label=r"$\|u_{ss} - U\|_\infty$")
plt.title("Convergencia hacia el Estado Estacionario - BT-CS")
plt.xlabel(r"$t$")
plt.ylabel(r"$error$")
plt.legend()
plt.tight_layout()
plt.show()